In [0]:
from pyspark.sql.types import StringType, DateType
import pyspark.sql.functions as F

In [0]:
df = spark.read.table("workspace.bronze.crm_prd_info")
display(df)

Validate string columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))


Extract category and product key from prod_key field

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(F.col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key"))))
display(df)

In [0]:
df.select("prd_line").distinct().show()

In [0]:
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.col("prd_line") == "R", "Road")
        .when(F.col("prd_line") == "M", "Mountain")
        .when(F.col("prd_line") == "T", "Touring")
        .when(F.col("prd_line") == "S", "Other Sales")
        .otherwise("n/a")
    )
)

In [0]:
display(df)

In [0]:
df = df.fillna({"prd_cost": 0})

In [0]:
df = df.withColumn("prd_start_dt", F.to_date(F.col("prd_start_dt"), "yyyy-MM-dd"))
df = df.withColumn("prd_end_dt", F.to_date(F.col("prd_end_dt"), "yyyy-MM-dd"))

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

df = df.withColumnsRenamed(RENAME_MAP)

In [0]:
display(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_products
LIMIT 10